In [7]:
from datasets import load_dataset, disable_progress_bar
from client import Client

client = Client()
client.clear_history()
disable_progress_bar()

Backend Online: LumiXAI Backend Online
Requesting database and file cleanup...
Database and result files cleared.


In [ ]:
# Load IMDB test split
dataset = load_dataset("imdb", split="test")

MAX_WORDS = 60
N_PER_CATEGORY = 8

def is_short(text):
    return len(text.split()) < MAX_WORDS

# Select short positive reviews
positive_samples = []
for review in dataset:
    if review["label"] == 1 and is_short(review["text"]):
        positive_samples.append(review["text"])
    if len(positive_samples) == N_PER_CATEGORY:
        break

# Select short negative reviews
negative_samples = []
for review in dataset:
    if review["label"] == 0 and is_short(review["text"]):
        negative_samples.append(review["text"])
    if len(negative_samples) == N_PER_CATEGORY:
        break

SAMPLES = []
for text in positive_samples:
    SAMPLES.append({"text": text, "category": "positive"})
for text in negative_samples:
    SAMPLES.append({"text": text, "category": "negative"})

print(f"Selected {len(SAMPLES)} samples:")
for i, s in enumerate(SAMPLES):
    preview = s["text"][:80].replace("\n", " ")
    print(f"[{i}] [{s['category'].upper()}] {preview}...")

Selected 16 samples:
[0] [POSITIVE] A stunningly well-made film, with exceptional acting, directing, writing, and ph...
[1] [POSITIVE] While the romance in this film is an important aspect, it is largely about the r...
[2] [POSITIVE] The theme is controversial and the depiction of the hypocritical and sexually st...
[3] [POSITIVE] This short deals with a severely critical writing teacher whose undiplomatic cri...
[4] [POSITIVE] A boy who adores Maurice Richard of the Montreal Canadiens receives, much to his...
[5] [POSITIVE] This story focuses on the birth defect known as FAS, or Fetal Alcohol Syndrome, ...
[6] [POSITIVE] Brilliant and moving performances by Tom Courtenay and Peter Finch....
[7] [POSITIVE] Extremely funny. More gags in each one of these episodes than in ten years of Fr...
[8] [NEGATIVE] Widow hires a psychopath as a handyman. Sloppy film noir thriller which doesn't ...
[9] [NEGATIVE] A sprawling, overambitious, plotless comedy that has no dramatic center. It was ...
[1

In [9]:
CLASSIFIER_MODEL = "distilbert-base-uncased-finetuned-sst-2-english"

classification_batch = [
    {
        "source": "huggingface",
        "model": CLASSIFIER_MODEL,
        "attributor": "captum_ig",
        "prompt": s["text"]
    }
    for s in SAMPLES
]

print(f"Submitting {len(classification_batch)} classification jobs...")
results_cls = client.run_smart_batch(classification_batch, sort_strategy="none")
print("Classification batch completed.")

Submitting 16 classification jobs...

Starting Smart Batch: 16 jobs in 1 groups.
Strategy: None (Original Order)


Elaboration:   0%|          | 0/16 [00:00<?, ?it/s]


Configuration for model: 'distilbert-base-uncased-finetuned-sst-2-english' | Attributor: 'captum_ig'...

Requesting explanation for: 'A stunningly well-made film, with exceptional acti...'

Requesting explanation for: 'While the romance in this film is an important asp...'

Requesting explanation for: 'The theme is controversial and the depiction of th...'

Requesting explanation for: 'This short deals with a severely critical writing ...'

Requesting explanation for: 'A boy who adores Maurice Richard of the Montreal C...'

Requesting explanation for: 'This story focuses on the birth defect known as FA...'

Requesting explanation for: 'Brilliant and moving performances by Tom Courtenay...'

Requesting explanation for: 'Extremely funny. More gags in each one of these ep...'

Requesting explanation for: 'Widow hires a psychopath as a handyman. Sloppy fil...'

Requesting explanation for: 'A sprawling, overambitious, plotless comedy that h...'

Requesting explanation for: 'I found this mo

Elaboration:   6%|▋         | 1/16 [00:17<04:24, 17.64s/it]


Requesting explanation for: 'A mean spirited, repulsive horror film about 3 mur...'


Elaboration: 100%|██████████| 16/16 [00:30<00:00,  1.91s/it]


 Global batch completed!
Classification batch completed.


In [10]:
GENERATIVE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
TEMPLATE = "Classify the sentiment of the following review as exactly 'Positive' or 'Negative'. Do not output any other text. Review: '{}'"

generation_batch = [
    {
        "source": "huggingface",
        "model": GENERATIVE_MODEL,
        "attributor": "captum_ig",
        "prompt": TEMPLATE.format(s["text"]),
        "ignore_special_tokens": True
    }
    for s in SAMPLES
]

print(f"Submitting {len(generation_batch)} generation jobs...")
results_gen = client.run_smart_batch(generation_batch, sort_strategy="none")
print("Generation batch completed.")

Submitting 16 generation jobs...

Starting Smart Batch: 16 jobs in 1 groups.
Strategy: None (Original Order)


Elaboration:   0%|          | 0/16 [00:00<?, ?it/s]


Configuration for model: 'Qwen/Qwen2.5-0.5B-Instruct' | Attributor: 'captum_ig'...

Requesting explanation for: 'Classify the sentiment of the following review as ...'

Requesting explanation for: 'Classify the sentiment of the following review as ...'

Requesting explanation for: 'Classify the sentiment of the following review as ...'

Requesting explanation for: 'Classify the sentiment of the following review as ...'

Requesting explanation for: 'Classify the sentiment of the following review as ...'

Requesting explanation for: 'Classify the sentiment of the following review as ...'

Requesting explanation for: 'Classify the sentiment of the following review as ...'

Requesting explanation for: 'Classify the sentiment of the following review as ...'

Requesting explanation for: 'Classify the sentiment of the following review as ...'

Requesting explanation for: 'Classify the sentiment of the following review as ...'

Requesting explanation for: 'Classify the sentiment of the follow

Elaboration: 100%|██████████| 16/16 [42:41<00:00, 160.07s/it]


 Global batch completed!
Generation batch completed.


In [ ]:

VALID_LABELS = ["positive", "negative"]

valid_samples = []

for i, result in enumerate(results_gen):
    trace = result.get("payload", {}).get("scores", [])
    if not trace:
        continue

    # Scan all the generated tokens to find the label
    label_token = None
    label_step = None
    for step_idx, step in enumerate(trace):
        token = step.get("generated_token", "").strip().lower()
        if token in VALID_LABELS:
            label_token = token
            label_step = step_idx
            break

    if label_token is None:
        continue

    # Check true category
    true_category = SAMPLES[i]["category"]
    if label_token != true_category:
        print(f"[{i}] WRONG prediction — true: {true_category}, predicted: {label_token} — skipped")
        continue

    valid_samples.append({
        "index": i,
        "category": true_category,
        "label_token": label_token,
        "label_step": label_step,
        "probability": trace[label_step]["probability"],
        "text": SAMPLES[i]["text"]
    })

print(f"\nValid samples (correct label): {len(valid_samples)} / {len(SAMPLES)}\n")
for s in valid_samples:
    preview = s["text"][:70].replace("\n", " ")
    print(
        f"[{s['index']}] [{s['category'].upper()}] "
        f"→ '{s['label_token']}' at step {s['label_step']} "
        f"(p={s['probability']:.2f}) | {preview}..."
    )

[2] WRONG prediction — true: positive, predicted: negative — skipped
[12] WRONG prediction — true: negative, predicted: positive — skipped

Valid samples (correct label): 14 / 16

[0] [POSITIVE] → 'positive' at step 0 (p=0.90) | A stunningly well-made film, with exceptional acting, directing, writi...
[1] [POSITIVE] → 'positive' at step 0 (p=0.82) | While the romance in this film is an important aspect, it is largely a...
[3] [POSITIVE] → 'positive' at step 0 (p=0.91) | This short deals with a severely critical writing teacher whose undipl...
[4] [POSITIVE] → 'positive' at step 0 (p=0.93) | A boy who adores Maurice Richard of the Montreal Canadiens receives, m...
[5] [POSITIVE] → 'positive' at step 0 (p=0.91) | This story focuses on the birth defect known as FAS, or Fetal Alcohol ...
[6] [POSITIVE] → 'positive' at step 0 (p=0.98) | Brilliant and moving performances by Tom Courtenay and Peter Finch....
[7] [POSITIVE] → 'positive' at step 8 (p=0.98) | Extremely funny. More gags in each o

In [12]:
client.free_memory()

Requesting VRAM cleanup...
Cleaned up memory and VRAM. Model unloaded.


{'status': 'success', 'message': 'Cleaned up memory and VRAM. Model unloaded.'}